# How far a second full-answer run moves the scores

Full-answer elicitation samples from the model, so a run is one draw. This
notebook reads how far the scores move when the draw is repeated, and turns that
into the resolution limit of every comparison the paper makes. Next-token
probabilities are deterministic and are not resampled.

It reads the committed population tables and nothing else except, for the
retention reading, the retained cells of each repeated run. A condition enters
when it has at least two FA runs: the 18 conditions that cross the six locally
run models with the as-released, German and Mexican-Spanish variants, ten of
them with a third run. Terra and the Mixtral references have one run and stay
out.

Within a condition-question pair $i$ with $k_i$ runs and standard deviation
$s_i$, the pooled run-to-run deviation weights each pair by its degrees of
freedom,

$$s_{\mathrm{pooled}} = \sqrt{\frac{\sum_i (k_i - 1)\, s_i^2}{\sum_i (k_i - 1)}},$$

which is $|x_1 - x_2| / \sqrt{2}$ averaged in square when every pair has two
runs. The between-condition deviation is the first run's standard deviation
across the 18 conditions within a question, root-mean-squared over the four
questions, and $s_{\mathrm{pooled}}^2$ over its square is the run-to-run share
of the variance; one minus that share is the intraclass correlation.

In [1]:
from culture.fidelity import GROUPS_TABLE, OVERALL_TABLE, PAIRED_TABLE
from culture.fidelity_baseline import manuscript_runs
from culture.fidelity_replicates import (
    REPLICATES_PAIRED_TABLE,
    REPLICATES_READINGS_TABLE,
    REPLICATES_TABLE,
    check_replicates,
    paired_spread,
    repeated_runs,
    replicate_readings,
    replicate_spread,
    retained_cells,
)
from culture.tables import read_csv
from machine_bias_reproduction.io_utils import write_csv

## Build

In [2]:
groups = read_csv(GROUPS_TABLE)
overall = read_csv(OVERALL_TABLE)
paired = read_csv(PAIRED_TABLE)
assert groups is not None and overall is not None and paired is not None
retained = retained_cells(repeated_runs(manuscript_runs(every_replicate=True), groups))
spread = replicate_spread(groups)
readings = replicate_readings(groups, overall, retained)
changes = paired_spread(paired)
print(len(spread), "spread rows,", len(readings), "readings,", len(changes), "paired scopes")

6 spread rows, 70 readings, 4 paired scopes


## Checks

The per-question rows must add up to the pooled row in pairs and in degrees of
freedom, every pair holds two or three runs, every bootstrap interval holds its
own estimate, and the four paired scopes rest on the same comparisons.

In [3]:
check_replicates(spread, readings, changes)
print("the replicate tables are consistent")

the replicate tables are consistent


## The tables

`population_fidelity_replicates.csv` holds one row per question, an `all` row
pooling the 72 pairs, and a `between conditions` row for scale: the pooled
deviation of accuracy, adaptability, structure, PFS and center alignment, the
degrees of freedom, and the Spearman correlation between the PFS orderings of
runs 1 and 2 and, over the three-run conditions, the range of it across run
pairs.

`population_fidelity_replicates_readings.csv` is one `reading, value` row per
number the text quotes: the design counts, the bootstrap interval of each pooled
deviation (pairs resampled 20,000 times from `GLOBAL_SEED`), the variance share
and intraclass correlation, the distribution of run-to-run PFS differences, the
zero floor, the stability of the limiting component and of the compression
reading, cell retention, the subgroup scores, and the cross-question summary.

`population_fidelity_replicates_paired.csv` reads the paired fine-tuning
changes of `population_fidelity_paired.csv` across its two FA runs, on all
retained cells and on the target country's cells: the mean change in PFS and in
center alignment per run, the pooled per-comparison deviation, and the number of
comparisons whose sign is the same in both runs.

In [4]:
write_csv(spread, REPLICATES_TABLE)
write_csv(readings, REPLICATES_READINGS_TABLE)
write_csv(changes, REPLICATES_PAIRED_TABLE)
value = readings.set_index("reading")["value"]
print(
    f"PFS moves by {value['pfs_sd']:.3f} [{value['pfs_sd_low']:.3f}, {value['pfs_sd_high']:.3f}]"
    f" between runs over {value['n_pairs']:.0f} pairs, {value['pfs_variance_share']:.1%} of its spread"
)

PFS moves by 0.029 [0.022, 0.037] between runs over 72 pairs, 3.5% of its spread


## Reading the tables

Accuracy and center alignment are read off individual cells and barely move.
Structure and PFS move more, and most of what PFS does move comes from pairs
whose structure correlation crosses the clip at zero, where the geometric mean
jumps to or from exactly zero; the zero-floor readings separate those pairs out.
The paired table is the one place where the run-to-run deviation is as large as
the effect: on the target country's cells alone, the mean change in PFS is not
separable from resampling, while the change in center alignment is.